In [15]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [16]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
786,Oh boy! Oh boy! On the cover of worn out VHS h...,negative
591,"The best thing about the movie is the name, as...",negative
824,I normally do not take the time to make commen...,negative
28,"Uzak (2002), a Turkish film shown in the U.S. ...",positive
244,"If you want a complete waste of time, because ...",negative


In [17]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [18]:
df = normalize_text(df)
df.head()

,review,sentiment
786,oh boy oh boy cover worn vhs picture sandra bu...,negative
591,best thing movie name describes plot acting le...,negative
824,normally take time make comment people read mo...,negative
28,uzak turkish film shown u s distant br br wa d...,positive
244,want complete waste time pulling lint belly bu...,negative


In [19]:
df['sentiment'].value_counts()

sentiment
negative    254
positive    246
Name: count, dtype: int64

In [20]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [21]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
786,oh boy oh boy cover worn vhs picture sandra bu...,0
591,best thing movie name describes plot acting le...,0
824,normally take time make comment people read mo...,0
28,uzak turkish film shown u s distant br br wa d...,1
244,want complete waste time pulling lint belly bu...,0


In [22]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [23]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [25]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/shettymanju2003/MLOPS.mlflow')
dagshub.init(repo_owner='shettymanju2003', repo_name='MLOPS', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


2026-02-21 16:51:06,612 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/shettymanju2003/MLOPS "HTTP/1.1 200 OK"


Initialized MLflow to track repo "shettymanju2003/MLOPS"

2026-02-21 16:51:06,613 - INFO - Initialized MLflow to track repo "shettymanju2003/MLOPS"


Repository shettymanju2003/MLOPS initialized!

2026-02-21 16:51:06,613 - INFO - Repository shettymanju2003/MLOPS initialized!


<Experiment: artifact_location='mlflow-artifacts:/a856dcdda10d4dbf8eed7ab19da212c6', creation_time=1771672628195, experiment_id='0', last_update_time=1771672628195, lifecycle_stage='active', name='Logistic Regression Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [26]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.20)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-02-21 16:51:07,543 - INFO - Starting MLflow run...
2026-02-21 16:51:08,061 - INFO - Logging preprocessing parameters...
2026-02-21 16:51:09,053 - INFO - Initializing Logistic Regression model...
2026-02-21 16:51:09,053 - INFO - Fitting the model...
2026-02-21 16:51:09,067 - INFO - Model training complete.
2026-02-21 16:51:09,067 - INFO - Logging model parameters...
2026-02-21 16:51:09,483 - INFO - Making predictions...
2026-02-21 16:51:09,483 - INFO - Calculating evaluation metrics...
2026-02-21 16:51:09,497 - INFO - Logging evaluation metrics...
2026-02-21 16:51:10,854 - INFO - Saving and logging the model...
2026/02/21 16:51:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 16:51:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run flawless-stag-327 at: https://dagshub.com/shettymanju2003/MLOPS.mlflow/#/experiments/0/runs/b64f7bf44f694b14987add5a8822639a
🧪 View experiment at: https://dagshub.com/shettymanju2003/MLOPS.mlflow/#/experiments/0
